# 02.3 — Loading `.mat` files

Your data lives in `.mat` files written by MATLAB. This notebook covers the three Python readers — `scipy.io.loadmat`, `mat73`, and `h5py` — when each one works, when each one fails, and how this codebase's `load_mat()` wrapper auto-detects the right one.

This is one of the highest-payoff notebooks in the curriculum: getting `.mat` I/O wrong produces confusing errors ("Please use HDF reader for matlab v7.3 files") or, worse, silently transposed data.

**Prerequisite:** [02.1 numpy vs MATLAB arrays](02.1_numpy_vs_matlab_arrays.ipynb).

## Section 1 — What MATLAB does

MATLAB's `save` command writes `.mat` files in one of several on-disk formats, and **the format depends on flags and file size**:

```matlab
save('data.mat', 'Data')            % v7 by default (compressed, pre-HDF5)
save('data.mat', 'Data', '-v7.3')   % v7.3 — HDF5-based, required for variables > 2 GB
save('data.mat', 'Data', '-v6')     % v6 — old, uncompressed
```

The catch: **v7.3 files are a completely different container** (HDF5) than v5/v6/v7 (MATLAB's proprietary binary). A Python reader built for one cannot read the other. MATLAB's own `load` hides this; Python makes you care.

The neural-data pipeline's `Decision_Data_*.mat` files are v7.3 (they're written with `-v7.3` because sessions can exceed 2 GB).

## Section 2 — The Python concepts you need

### 2.1 — The three readers

| Reader | Handles | Returns | Fails on |
|---|---|---|---|
| `scipy.io.loadmat` | v4, v5, v6, v7 | dict of numpy arrays | v7.3 (raises `NotImplementedError: Please use HDF reader...`) |
| `mat73.loadmat` | v7.3 only | dict of numpy arrays / nested dicts | pre-v7.3 (raises) |
| `h5py.File` | v7.3 (raw HDF5) | h5py Dataset objects (lazy) | pre-v7.3; needs manual dereferencing for structs/cells |

**Rule of thumb:** use `scipy.io` for old files, `mat73` for v7.3 files with structs, `h5py` directly only when you need lazy/partial reads of huge arrays.

### 2.2 — How to tell which format a file is

The `.mat` header tells you. Here's the actual byte layout — this diagram matters because the codebase's format detector reads exactly these offsets:

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(12, 3.2))

# Byte-layout bands for a v7.3 MAT file header
bands = [
    (0, 116, "ASCII description text\n'MATLAB 7.3 MAT-file, Platform: ...'", "#cce4ff"),
    (116, 124, "subsys\noffset", "#e6e6e6"),
    (124, 126, "version\n0x0200", "#ffcccc"),
    (126, 128, "endian\n'IM'", "#ffe4cc"),
    (128, 512, "(zero padding)", "#f5f5f5"),
    (512, 620, "HDF5 magic + embedded HDF5 stream\n'\\x89HDF\\r\\n...'", "#e6f0d0"),
]
total = 620
for start, end, label, color in bands:
    w = (end - start) / total * 12
    x = start / total * 12
    rect = mpatches.FancyBboxPatch((x, 0.8), w, 1.2, boxstyle="round,pad=0.01",
        facecolor=color, edgecolor="black", linewidth=1.0)
    ax.add_patch(rect)
    fs = 9 if (end - start) > 30 else 7
    ax.text(x + w / 2, 1.4, label, ha="center", va="center", fontsize=fs)
    ax.text(x, 0.55, str(start), ha="center", fontsize=8, color="#666")
ax.text(12, 0.55, "...", ha="center", fontsize=8, color="#666")

# Annotations
ax.annotate("scipy checks HERE (fails for v7.3:\nheader is ASCII, not MATLAB v5 tag)",
    xy=(0.8, 2.05), xytext=(1.2, 2.9), fontsize=9, color="#3366aa",
    arrowprops=dict(arrowstyle="->", color="#3366aa"))
ax.annotate("our detector reads the 2-byte\nversion word at offset 124\n(0x0200 = v7.3, 0x0100 = v5/6/7)",
    xy=(124 / total * 12 + 0.05, 2.05), xytext=(4.6, 2.9), fontsize=9, color="#aa0000",
    arrowprops=dict(arrowstyle="->", color="#aa0000"))
ax.annotate("the actual HDF5 stream\nstarts at offset 512",
    xy=(512 / total * 12 + 0.5, 2.05), xytext=(9.2, 2.9), fontsize=9, color="#447722",
    arrowprops=dict(arrowstyle="->", color="#447722"))

ax.set_xlim(-0.3, 12.5); ax.set_ylim(0.3, 3.6)
ax.axis("off")
ax.set_title("Byte layout of a MATLAB v7.3 .mat file header (offsets in bytes)", fontsize=12)
plt.tight_layout(); plt.show()

**The subtle part:** a v7.3 file does NOT start with the HDF5 magic bytes — it starts with a 116-byte ASCII text header, and the HDF5 stream begins at offset 512. So the naive check "does the file start with `\x89HDF`?" fails for real MATLAB files. The reliable discriminator is the **2-byte version word at offset 124**: `0x0200` means v7.3, `0x0100` means v5/6/7.

(This exact bug existed in an early version of this codebase's loader — it checked offset 0 and misclassified every real v7.3 file. The fix is in `src/neural_data_decoding/data/mat_files.py`.)

### 2.3 — Reading a pre-v7.3 file with scipy

In [ ]:
import numpy as np
import scipy.io as sio

# Write a small v5-format file (scipy always writes pre-v7.3).
# tempfile.gettempdir() = /tmp on Unix, %TEMP% on Windows — cross-platform.
import tempfile
from pathlib import Path
DEMO_V5 = Path(tempfile.gettempdir()) / "demo_v5.mat"
sio.savemat(DEMO_V5, {"Data": np.arange(12).reshape(3, 4), "label": "hello"})

loaded = sio.loadmat(DEMO_V5)
print(sorted(loaded.keys()))         # your variables + __header__/__version__/__globals__
print(loaded["Data"])
print(loaded["Data"].shape)

Notes:

* scipy adds `__header__`, `__version__`, `__globals__` metadata keys — ignore them.
* **Everything comes back 2-D at minimum.** MATLAB has no true 1-D arrays, so a MATLAB row vector loads as shape `(1, N)` and a scalar as `(1, 1)`. Use `squeeze_me=True` to collapse singleton dims — but beware: that's the same over-eager squeezing that caused this project's CorrectTrial CM-table bug. Squeeze deliberately, not globally.
* Strings come back as numpy `str_` arrays; call `str(...)` to normalize.

### 2.4 — Reading a v7.3 file with mat73

The development machine keeps a real v7.3 fixture — one trial of Decision-epoch neural data — under `results/Decision/` (gitignored: [00.6](../00_orientation/00.6_git_and_github_for_matlab_users.ipynb) explains why 8 MB of binary data stays out of history). The cells below load it when present and print a note when it isn't:

In [ ]:
import mat73
from pathlib import Path

FIXTURE_DIR = Path("../../results/Decision")
DATA_FIXTURE = FIXTURE_DIR / "Decision_Data_0000011.mat"
HAVE_FIXTURE = DATA_FIXTURE.is_file()

if HAVE_FIXTURE:
    d = mat73.loadmat(DATA_FIXTURE)
    print(list(d.keys()))
    Data = d["Data"]
    print(type(Data), Data.dtype, Data.shape)   # (58, 3001, 6) = (C, TT, A) per 02.2
else:
    print("fixture not present on this machine — skipping (the notebook still teaches; outputs just stay empty)")

In [ ]:
# The paired Target file is a struct — mat73 returns it as a nested dict.
if HAVE_FIXTURE:
    t = mat73.loadmat(FIXTURE_DIR / "Target_0000011.mat")
    Target = t["Target"]
    print(type(Target))
    print(f"{len(Target)} fields; a few of them:")
    for key in ["SessionName", "SelectedObjectDimVals", "CorrectTrial", "Dimensionality"]:
        print(f"  {key} = {Target[key]!r}")
else:
    print("fixture not present — skipping")

**mat73's struct handling is why we prefer it over raw h5py:** a MATLAB struct becomes a plain Python dict; cell arrays become lists. With h5py you'd have to dereference HDF5 object references by hand.

**NaN survives the round trip.** The neural data uses NaN for removed channels; check:

In [ ]:
import numpy as np
if HAVE_FIXTURE:
    print(f"NaN count: {np.isnan(Data).sum()} of {Data.size} values")
    print(f"NaN per area: {[int(np.isnan(Data[:, :, a]).sum()) for a in range(Data.shape[2])]}")
else:
    print("fixture not present — skipping")

### 2.5 — When to reach for h5py directly

For a 10 GB session file where you only need one slice, `mat73` loads everything into RAM. `h5py` gives you lazy access:

```python
import h5py

with h5py.File("huge_session.mat", "r") as f:
    print(list(f.keys()))              # variable names
    dset = f["Data"]                    # NO data read yet — lazy handle
    chunk = dset[0:10, :, 0]            # reads ONLY this slice from disk
```

Caveats: h5py sees the raw HDF5 layout, where MATLAB stores arrays **transposed** (column-major on disk); you may need `.T` or explicit `np.transpose` to match what MATLAB/`mat73` shows you. Structs appear as HDF5 groups; cell arrays as object-reference datasets you must dereference. For this codebase's file sizes (one trial per file), `mat73` is the right default and h5py is unnecessary.

### 2.6 — The codebase's `load_mat()` — auto-detection

`src/neural_data_decoding/data/mat_files.py` wraps the decision in a single entry point. The decision tree:

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(9, 5))

def node(x, y, w, label, color, fs=10):
    rect = mpatches.FancyBboxPatch((x - w / 2, y - 0.32), w, 0.64,
        boxstyle="round,pad=0.06", facecolor=color, edgecolor="black", linewidth=1.2)
    ax.add_patch(rect)
    ax.text(x, y, label, ha="center", va="center", fontsize=fs, family="monospace")

def edge(x1, y1, x2, y2, label=""):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
        arrowprops=dict(arrowstyle="->", color="black", lw=1.3))
    if label:
        ax.text((x1 + x2) / 2 + 0.35, (y1 + y2) / 2, label, fontsize=10,
            color="#aa0000", fontweight="bold")

node(4.5, 5.0, 3.4, "load_mat(path)", "#dddddd", fs=11)
node(4.5, 3.6, 5.2, "read 2 bytes at offset 124", "#ffe4cc", fs=10)
node(1.8, 1.9, 3.2, "scipy.io.loadmat\n(v5 / v6 / v7)", "#cce4ff")
node(7.2, 1.9, 3.2, "mat73.loadmat\n(v7.3 / HDF5)", "#e6f0d0")

edge(4.5, 4.65, 4.5, 4.0)
edge(3.4, 3.3, 2.1, 2.3, "0x0100")
edge(5.6, 3.3, 6.9, 2.3, "0x0200")

ax.text(1.8, 0.9, "returns dict incl.\n__header__ etc.", ha="center", fontsize=9, color="#666")
ax.text(7.2, 0.9, "returns dict of\nuser variables only", ha="center", fontsize=9, color="#666")

ax.set_xlim(-0.2, 9.5); ax.set_ylim(0.4, 5.7)
ax.axis("off")
ax.set_title("load_mat() format auto-detection", fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# Use it — same call regardless of format:
from neural_data_decoding.data.mat_files import load_mat

if HAVE_FIXTURE:
    d = load_mat(DATA_FIXTURE)          # v7.3 → routed to mat73
    print("v7.3 fixture:", d["Data"].shape)

d2 = load_mat(DEMO_V5)                   # v5 → routed to scipy
print("v5 demo    :", d2["Data"].shape)

## Section 3 — The `neural_data_decoding` implementation

The detector is 15 lines. Read it:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/data/mat_files.py").read_text().splitlines()
start = next(i for i, line in enumerate(src) if line.startswith("_MAT_VERSION_OFFSET"))
end = next(i for i, line in enumerate(src) if line.startswith("def load_mat"))
for i in range(start - 8, end):
    print(f"{i + 1:3} | {src[i]}")

Things to spot:

* Module-level constants `_MAT_VERSION_OFFSET = 124` and `_MAT_VERSION_V73 = 0x0200` — named numbers, not magic literals.
* `int.from_bytes(version_bytes, "little")` — the MATLAB header is little-endian on every platform you'll meet.
* The docstring records WHY offset 124 (and why not offset 0) — exactly the kind of comment that saves the next reader a debugging session.
* The leading-underscore names (`_is_hdf5_mat`) mark internals; the public API is just `load_mat`.

## Section 4 — Hands-on exercises

### Exercise 4.1 — inspect a header by hand

Read the first 130 bytes of the v7.3 fixture and print (a) the ASCII description, (b) the version word as hex.

In [ ]:
# Your attempt here


In [ ]:
# Reveal:
if HAVE_FIXTURE:
    raw = DATA_FIXTURE.read_bytes()[:130]
    print("description:", raw[:116].decode("ascii", errors="replace").strip())
    version = int.from_bytes(raw[124:126], "little")
    print(f"version word: 0x{version:04x}  ({'v7.3 (HDF5)' if version == 0x0200 else 'pre-v7.3'})")
else:
    print("fixture not present — try this on any -v7.3 .mat file you have")

### Exercise 4.2 — trigger the classic error

Load the v7.3 fixture with the WRONG reader (`scipy.io.loadmat`) and observe the exact error message. This is the error you'll recognize instantly in the wild.

In [ ]:
import scipy.io as sio
if HAVE_FIXTURE:
    try:
        sio.loadmat(DATA_FIXTURE)
    except NotImplementedError as e:
        print(f"NotImplementedError: {e}")
else:
    print("fixture not present — the error you would see:")
    print("NotImplementedError: Please use HDF reader for matlab v7.3 files, e.g. h5py")

## Section 5 — Common errors

### `NotImplementedError: Please use HDF reader for matlab v7.3 files`

You used `scipy.io.loadmat` on a v7.3 file. Switch to `mat73.loadmat` — or just use the project's `load_mat()`, which auto-detects.

### `TypeError: ... is not a MATLAB 7.3 file` (from mat73)

The reverse: mat73 on a pre-v7.3 file. Same fix — `load_mat()`.

### Shapes look transposed after loading with h5py

MATLAB stores column-major; h5py shows you the on-disk layout, so a MATLAB `(58, 3001, 6)` array appears as `(6, 3001, 58)`. `mat73` un-transposes for you; raw h5py does not.

### `KeyError: 'Data'` after loading

The variable name inside the file isn't what you expected. Print `list(loaded.keys())` — the name is whatever the MATLAB `save` call used, and scipy adds `__header__`-style keys you should skip.

### Struct fields come back as 0-d object arrays (scipy)

scipy's default struct handling wraps fields in object arrays. Either pass `struct_as_record=False, squeeze_me=True` (careful — global squeeze), or handle the unwrapping explicitly like `_normalize_target_struct` in `src/neural_data_decoding/data/mat_dataset.py`.

### Loading is slow for many small files

Each `loadmat` call re-opens the file. For per-trial datasets (like this project's), that's the intended access pattern — the `Dataset` loads lazily per `__getitem__`, so training never holds all trials in RAM. If you genuinely need everything, load once and cache (see `preload=True` on `MatFileTrialDataset`).

## Section 6 — Further reading

- [scipy.io.loadmat docs](https://docs.scipy.org/doc/scipy/reference/generated/scipy.io.loadmat.html) — every keyword, including `squeeze_me` and `struct_as_record`.
- [mat73 on GitHub](https://github.com/skjerns/mat7.3) — small library, readable source.
- [h5py docs](https://docs.h5py.org/) — for the lazy-read case.
- [MAT-file format spec (PDF)](https://www.mathworks.com/help/pdf_doc/matlab/matfile_format.pdf) — MathWorks' official byte-level documentation; documents the 128-byte header including the version word at offset 124.
- [`src/neural_data_decoding/data/mat_files.py`](../../src/neural_data_decoding/data/mat_files.py) — the production wrapper.

Next up: **[02.4 PyTorch tensors intro](02.4_pytorch_tensors_intro.ipynb)**.